In [3]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from langgraph.checkpoint.memory import InMemorySaver
from langchain_tavily import TavilySearch
from typing import Dict, Any
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
def web_search(query: str, max_results: int = 1, include_raw_content: bool = False) -> Dict[str, Any]:
    """A Web searching tool that fetches the content from web network"""
    return TavilySearch(
        max_results=min(max_results, 1),
        include_raw_content=False
    ).invoke(input=query)
    
research_subagent = {
    "name": "Research Subagent",
    "description": "A subagents designed to research and provide results for a query",
    "system_prompt": "You are a great Research Assistant",
    "tools": [web_search],
    "model": "groq:Openai/gpt-oss-20b"
}

In [3]:
d_agent = create_deep_agent(
    model="groq:Openai/gpt-oss-20b",
    backend = StateBackend(),
    subagents=[research_subagent],
    checkpointer=InMemorySaver(),
    system_prompt="""You are a financial coordinator. Delegate market research/live data tasks to 'Research Subagent'. Do not answer from memory. Keep responses short."""
)
result = d_agent.invoke({
    "messages": [
        {
        "role": "user", 
        "content": "research the market standards and determine whether this is the right time to buy golds"
        }
    ]
},
config={"configurable": {"thread_id": "10"}}
)
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

research the market standards and determine whether this is the right time to buy golds
================================== Ai Message ==================================
Tool Calls:
  task (fc_13290726-2837-4d5a-ad71-ae32c1bd9599)
 Call ID: fc_13290726-2837-4d5a-ad71-ae32c1bd9599
  Args:
    description: Research current market standards and indicators to evaluate if it's a favorable time to buy gold.
    subagent_type: Research Subagent
================================= Tool Message =================================
Name: task

Below is a concise, up‑to‑date snapshot (mid‑October 2024) of the key market standards and technical‑fundamental indicators that analysts and investors use when judging whether it’s a good time to buy gold.  The information is drawn from publicly available data, central‑bank statements, and widely‑used technical tools.  If you need deeper dives into any particular indicator, let me

In [5]:
from pydantic import BaseModel, Field
from deepagents import create_deep_agent

class ResearchFinding(BaseModel):
    """Structured findings from a research task. """
    summary: str = Field(description="Summary of findings")
    confidence: float = Field(description="Confidence score from 0 to 1")
    sources: list[str] = Field(description="List of source URLs") 

research_subagent = {
    "name": "researcher",
    "description": "Researches topics and returns structured findings",
    "system_prompt": "Research the given topic thoroughly. Return your findings.",
    "tools": [web_search],
    "response_format": ResearchFinding,
}

deepagent_structured = create_deep_agent(
    model="groq:Openai/gpt-oss-20b",
    subagents=[research_subagent],
)

result = deepagent_structured.invoke({
    "messages": [
        {
        "role": "user", 
        "content": "research the market standards and determine whether this is the right time to buy golds"
        }
    ]
},
config={"configurable": {"thread_id": "10"}}
)
for m in result["messages"]:
    m.pretty_print()

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m0fnmyjte36bwmy5y9ytzy57` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197901, Requested 2992. Please try again in 6m25.776s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}